In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("Shape:", df.shape)

Shape: (7043, 21)


In [2]:
df = df.drop(columns=['customerID', 'gender'])

print("Shape after dropping:", df.shape)
print("Remaining columns:", df.columns.tolist())

Shape after dropping: (7043, 19)
Remaining columns: ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [3]:
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Churn value counts:")
print(df['Churn'].value_counts())
print("\nChurn dtype:", df['Churn'].dtype)

Churn value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn dtype: int64


In [4]:
# Customers with high charges but short tenure = high risk
df['charges_to_tenure_ratio'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# How many additional services is the customer using?
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
            'TechSupport', 'StreamingTV', 'StreamingMovies']
df['service_adoption_score'] = (df[services] == 'Yes').sum(axis=1)

# Tenure groups - new/developing/established/loyal
df['tenure_group'] = pd.cut(df['tenure'], 
                             bins=[0, 12, 24, 48, 72],
                             labels=['New', 'Developing', 'Established', 'Loyal'],
                             include_lowest=True)

# Is the customer on the most dangerous combination?
df['high_risk_combo'] = (
    (df['Contract'] == 'Month-to-month') & 
    (df['MonthlyCharges'] > 65)
).astype(int)

print("New features added!")
print(df[['charges_to_tenure_ratio', 'service_adoption_score', 
          'tenure_group', 'high_risk_combo']].head(10))

New features added!
   charges_to_tenure_ratio  service_adoption_score tenure_group  \
0                14.925000                       1          New   
1                 1.627143                       2  Established   
2                17.950000                       2          New   
3                 0.919565                       3  Established   
4                23.566667                       0          New   
5                11.072222                       3          New   
6                 3.873913                       2   Developing   
7                 2.704545                       1          New   
8                 3.613793                       4  Established   
9                 0.891270                       2        Loyal   

   high_risk_combo  
0                0  
1                0  
2                0  
3                0  
4                1  
5                1  
6                1  
7                0  
8                1  
9                0  


In [5]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 
               'PaperlessBilling', 'SeniorCitizen']

# These are already Yes/No or 0/1
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in yes_no_cols:
    df[col] = (df[col] == 'Yes').astype(int)

print("Binary columns encoded")
print(df[binary_cols].head())

Binary columns encoded
   Partner  Dependents  PhoneService  PaperlessBilling  SeniorCitizen
0        1           0             0                 1              0
1        0           0             1                 0              0
2        0           0             1                 1              0
3        0           0             0                 0              0
4        0           0             1                 1              0


In [6]:
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract', 
              'PaymentMethod', 'tenure_group']

df = pd.get_dummies(df, columns=multi_cols, drop_first=False)

print("Shape after encoding:", df.shape)
print("Columns:", df.columns.tolist())

Shape after encoding: (7043, 47)
Columns: ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'charges_to_tenure_ratio', 'service_adoption_score', 'high_risk_combo', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credi

In [7]:
cols_to_drop = [
    # Multicollinearity with tenure
    'TotalCharges',
    
    # Dummy variable trap - drop one category from each group
    'MultipleLines_No',
    'InternetService_DSL',
    'OnlineSecurity_No',
    'OnlineBackup_No',
    'DeviceProtection_No',
    'TechSupport_No',
    'StreamingTV_No',
    'StreamingMovies_No',
    'Contract_Month-to-month',
    'PaymentMethod_Mailed check',
    'tenure_group_New'
]

df = df.drop(columns=cols_to_drop)

print("Shape after dropping redundant columns:", df.shape)
print("Final columns:", df.columns.tolist())

Shape after dropping redundant columns: (7043, 35)
Final columns: ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'Churn', 'charges_to_tenure_ratio', 'service_adoption_score', 'high_risk_combo', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'tenure_group_Developing', 'tenure_group_Established', 'tenure_group_Loyal']


In [8]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['tenure', 'MonthlyCharges', 
                  'charges_to_tenure_ratio', 'service_adoption_score']

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print("After scaling:")
print(df[numerical_cols].describe().round(3))

After scaling:
         tenure  MonthlyCharges  charges_to_tenure_ratio  \
count  7043.000        7043.000                 7043.000   
mean     -0.000          -0.000                   -0.000   
std       1.000           1.000                    1.000   
min      -1.318          -1.546                   -0.631   
25%      -0.952          -0.973                   -0.518   
50%      -0.137           0.186                   -0.424   
75%       0.921           0.834                    0.020   
max       1.614           1.794                    8.608   

       service_adoption_score  
count                7043.000  
mean                    0.000  
std                     1.000  
min                    -1.103  
25%                    -1.103  
50%                    -0.021  
75%                     0.521  
max                     2.145  


In [9]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X.to_csv('../data/X_processed.csv', index=False)
y.to_csv('../data/y_processed.csv', index=False)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Churn rate preserved:", y.mean().round(4))

Feature matrix shape: (7043, 34)
Target shape: (7043,)
Churn rate preserved: 0.2654


In [11]:
print("FEATURE ENGINEERING SUMMARY")
print(f"Original features:        21")
print(f"After dropping useless:   19")
print(f"After encoding:           47")
print(f"After dropping redundant: 35")
print(f"Final feature matrix:     {X.shape[1]} features, {X.shape[0]} rows")

print("\nNew features created:")
print("  - charges_to_tenure_ratio")
print("  - service_adoption_score")
print("  - tenure_group (binned)")
print("  - high_risk_combo")
print("\nDropped features:")
print("  - customerID (identifier)")
print("  - gender (zero signal)")
print("  - TotalCharges (0.83 correlation with tenure)")

FEATURE ENGINEERING SUMMARY
Original features:        21
After dropping useless:   19
After encoding:           47
After dropping redundant: 35
Final feature matrix:     34 features, 7043 rows

New features created:
  - charges_to_tenure_ratio
  - service_adoption_score
  - tenure_group (binned)
  - high_risk_combo

Dropped features:
  - customerID (identifier)
  - gender (zero signal)
  - TotalCharges (0.83 correlation with tenure)
